# Source-supported posterior: finite witnesses

See [theory_main.md](theory_main.md). This notebook verifies algebra and the common-view identification counterexample. It does not train an HSE/LLapDiff model or evaluate industrial diagnosis.

In [ ]:
import csv, os
from pathlib import Path
import numpy as np
rows=[]
def record(name,value):
    rows.append(dict(witness=name,value=float(value),scope='finite_theory_only'))
    print(name, float(value))
I=np.eye(4)
Oa=np.diag([1.,1.,0.,0.]); Ob=np.diag([1.,0.,1.,0.])
Us=np.diag([1.,1.,1.,0.]); Cs=np.diag([1.,0.,0.,0.])
blocks=[Cs,Oa-Cs,Us-Oa,I-Us]
np.testing.assert_allclose(sum(blocks),I)
for j,Pj in enumerate(blocks):
    np.testing.assert_allclose(Pj@Pj,Pj)
    for k,Q in enumerate(blocks):
        if j!=k: np.testing.assert_allclose(Pj@Q,0.)
record('source_partition_sum_residual',np.linalg.norm(sum(blocks)-I))
record('source_role_rank_each',min(np.trace(Pj) for Pj in blocks))
Ot=np.diag([0.,1.,1.,0.]); Ct=np.zeros((4,4))
np.testing.assert_allclose(sum([Ct,Ot-Ct,Us-Ot,I-Us]),I)
record('target_common_rank',np.trace(Ct))
record('target_missing_rank',np.trace(Us-Ot))

## Sensitivity and overlapping source views
A positive coordinate sensitivity is not individual recoverability. The second example uses two actual source pair laws, (C,P) and (C,M), not merely separate one-dimensional marginals.

In [ ]:
A=np.array([[1.,1.]])
a=np.array([2.,3.]); b=a+np.array([1.,-1.])
assert np.all(np.diag(A.T@A)>0)
np.testing.assert_allclose(A@a,A@b)
record('mixed_operator_ambiguous_state_distance',np.linalg.norm(a-b))
record('mixed_operator_observation_difference',np.linalg.norm(A@a-A@b))
# Axis order is (C,P,M). C is independent of the fair pair's generating bit.
pm_plus=np.eye(2)/2; pm_minus=np.fliplr(np.eye(2))/2
joint_plus=np.stack([pm_plus/2,pm_plus/2])
joint_minus=np.stack([pm_minus/2,pm_minus/2])
for joint in [joint_plus,joint_minus]:
    np.testing.assert_allclose(joint.sum(),1.)
# Source A sees (C,P), source B sees (C,M): both observed joint laws coincide.
np.testing.assert_allclose(joint_plus.sum(axis=2),joint_minus.sum(axis=2))
np.testing.assert_allclose(joint_plus.sum(axis=1),joint_minus.sum(axis=1))
# Their common-view conditional p(M|C) agrees, but the full-input conditional does not.
np.testing.assert_allclose(joint_plus.sum(axis=1)/.5,joint_minus.sum(axis=1)/.5)
conditional_plus=joint_plus[1,1,1]/joint_plus[1,1,:].sum()
conditional_minus=joint_minus[1,1,1]/joint_minus[1,1,:].sum()
conditional_gap=abs(conditional_plus-conditional_minus)
assert conditional_gap==1
record('unpaired_same_marginal_conditional_gap',conditional_gap)

## Source-null likelihood does not imply prior independence

In [ ]:
rho=.8; prior=np.array([[1.,rho],[rho,1.]])
operator=np.array([[1.,0.]])
gain=prior@operator.T/(float((operator@prior@operator.T).item())+1)
mean=gain[:,0]
post=prior-gain@operator@prior
np.testing.assert_allclose(mean,[.5,.4])
np.testing.assert_allclose(post[1,1],.68)
record('correlated_prior_null_posterior_mean',mean[1])
record('correlated_prior_null_posterior_variance',post[1,1])
independent=np.eye(2); gain_i=independent@operator.T/2
post_i=independent-gain_i@operator@independent
record('independent_prior_null_posterior_variance',post_i[1,1])
assert post_i[1,1]==1

## Projection preserves scope, not posterior correctness

In [ ]:
G=np.diag([0.,0.,1.,0.]); observed=np.array([.4,-.3,0.,0.])
rng=np.random.default_rng(18); v=G@rng.normal(size=4)
leak=[]; drift=[]
for k in range(100):
    v=G@(.9*v+rng.normal(size=4))
    state=observed+v
    leak.append(np.linalg.norm((I-G)@v))
    drift.append(np.linalg.norm(Oa@(state-observed)))
record('projected_update_max_forbidden_energy',max(leak))
record('projected_update_max_observed_drift',max(drift))
assert max(leak)==0 and max(drift)==0
G0=np.zeros((4,4))
recovered=None if np.linalg.matrix_rank(G0)==0 else G0@rng.normal(size=4)
assert recovered is None
record('empty_eligibility_generated_coordinates',0)
# A target can reverse the coupling even after the source joint has been identified.
record('target_conditional_reversal_gap',abs(conditional_plus-conditional_minus))

## Finite outputs
The values remain comparable to the retained 14-row witness. The stronger overlapping-source example has the same conditional gap, not a new empirical performance score.

In [ ]:
out=Path(os.environ.get('TII_BUILD_OUTPUT','outputs/tii_support'))
out.mkdir(parents=True,exist_ok=True)
with (out/'support_witness.csv').open('w',newline='',encoding='utf-8') as f:
    writer=csv.DictWriter(f,fieldnames=['witness','value','scope'])
    writer.writeheader(); writer.writerows(rows)
print('SUPPORT_POSTERIOR_FINITE_WITNESS_PASS',len(rows))

## Method witnesses: objective, reverse update and distributional boundaries

The following finite computations accompany Chapters 2–3. They do not fit a denoiser or evaluate an industrial recording. The oracle velocity in the update check uses known truth deliberately: it tests the stated algebra, not learned posterior correctness. Original support results above are unchanged.

In [ ]:
method_rows=[]
def method_record(name,value):
    method_rows.append(dict(witness=name,value=float(value),scope='finite_method_only'))
    print(name, float(value))

# Standard velocity conversion and its actual clean-target weighting.
w0=np.array([.3,-.8,.5]); eps=np.array([-.2,.7,1.1])
delta=np.array([.1,-.3,.2])
conversion_error=[]; weight_error=[]
for angle in [.2,.8,1.4]:
    alpha,sigma=np.cos(angle),np.sin(angle)
    wk=alpha*w0+sigma*eps
    velocity=alpha*eps-sigma*w0
    np.testing.assert_allclose(alpha*wk-sigma*velocity,w0,atol=1e-12)
    np.testing.assert_allclose(sigma*wk+alpha*velocity,eps,atol=1e-12)
    conversion_error.append(np.max(np.abs(alpha*wk-sigma*velocity-w0)))
    vhat=velocity+delta
    w0hat=alpha*wk-sigma*vhat
    weight_error.append(abs(np.mean(delta**2)-np.mean((w0hat-w0)**2)/sigma**2))
method_record('velocity_clean_conversion_max_error',max(conversion_error))
method_record('velocity_clean_weighted_loss_max_error',max(weight_error))
assert max(conversion_error)<1e-12 and max(weight_error)<1e-12


In [ ]:
# A genuinely non-diagonal basis, not a coordinate mask.
B=np.array([[1.],[1.],[0.]])/np.sqrt(2)
G=B@B.T
Bo=np.array([[1.],[-1.],[0.]])/np.sqrt(2)
Po=Bo@Bo.T
observed=(Bo*.7).ravel()
true_w=np.array([.6]); true_eps=np.array([-.4])
angles=[1.4,1.,.5,0.]
w=np.cos(angles[0])*true_w+np.sin(angles[0])*true_eps
path_error=[]; forbidden=[]; observed_drift=[]
for angle,next_angle in zip(angles[:-1],angles[1:]):
    alpha,sigma=np.cos(angle),np.sin(angle)
    velocity=alpha*true_eps-sigma*true_w
    clean=alpha*w-sigma*velocity
    noise=sigma*w+alpha*velocity
    w=np.cos(next_angle)*clean+np.sin(next_angle)*noise
    ambient=B@w
    path_error.append(np.max(np.abs(w-(np.cos(next_angle)*true_w+np.sin(next_angle)*true_eps))))
    forbidden.append(np.linalg.norm((np.eye(3)-G)@ambient))
    observed_drift.append(np.linalg.norm(Po@(observed+ambient)-observed))
method_record('intrinsic_ddim_oracle_path_max_error',max(path_error))
method_record('intrinsic_ddim_forbidden_max_norm',max(forbidden))
method_record('intrinsic_ddim_observed_drift_max_norm',max(observed_drift))
method_record('intrinsic_ddim_terminal_max_error',np.max(np.abs(w-true_w)))
assert max(path_error+forbidden+observed_drift)<1e-12


In [ ]:
# Joint score on a slice is not the marginal score.
# These are implied fixed-noise densities, not finite sampler output variances.
rho=.8
cov=np.array([[1.,rho],[rho,1.]])
precision=np.linalg.inv(cov)
slice_variance=1/precision[0,0]
marginal_variance=cov[0,0]
score_gap=abs(-precision[0,0]-(-1/marginal_variance)) # W=1,V=0
method_record('joint_score_slice_implied_variance',slice_variance)
method_record('true_marginal_variance',marginal_variance)
method_record('joint_slice_vs_marginal_score_gap_at_one',score_gap)
np.testing.assert_allclose(slice_variance,.36,atol=1e-12)
assert score_gap>1

# Exact covariance propagation for coherent versus independent marginal draws.
L=np.array([[1.,0.],[rho,np.sqrt(1-rho*rho)]])
coherent=L@L.T
independent=np.eye(2)
method_record('coherent_cross_covariance',coherent[0,1])
method_record('independent_cross_covariance',independent[0,1])
method_record('coherent_sum_variance',np.ones(2)@coherent@np.ones(2))
method_record('independent_sum_variance',np.ones(2)@independent@np.ones(2))
np.testing.assert_allclose(np.diag(coherent),np.diag(independent))
np.testing.assert_allclose(coherent,cov,atol=1e-12)
with (out/'method_witness.csv').open('w',newline='',encoding='utf-8') as f:
    writer=csv.DictWriter(f,fieldnames=['witness','value','scope'])
    writer.writeheader();writer.writerows(method_rows)
print('METHOD_FINITE_WITNESS_PASS',len(method_rows))
